# ACT Training on comma2k19

This notebook converts an extracted comma2k19 chunk into a local LeRobotDataset and trains an ACT policy with LeRobot's training CLI.

The conversion step is recommended because repeatedly streaming and decoding comma2k19 HEVC videos during training is inefficient.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
CHUNK_PATH = PROJECT_DIR / "comma2k19_data" / "extracted" / "Chunk_1"
DATASET_REPO_ID = "local/comma2k19_act"
DATASET_ROOT = PROJECT_DIR / "lerobot_datasets" / DATASET_REPO_ID
OUTPUT_DIR = PROJECT_DIR / "outputs" / "train" / "act_comma2k19"

print("Project:", PROJECT_DIR)
print("Chunk:", CHUNK_PATH)
print("Dataset root:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

Project: /content
Chunk: /content/comma2k19_data/extracted/Chunk_1
Dataset root: /content/lerobot_datasets/local/comma2k19_act
Output: /content/outputs/train/act_comma2k19


In [ ]:
!pip install lerobot

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.5 MB/s eta 0:00:00
   

## Optional Smoke-Test Conversion

This creates a tiny dataset with two episodes and a short frame cap. Run it first to verify paths and video encoding before converting the full chunk.

In [ ]:
%run ./comma2k19_FSD/dataset_setup.ipynb

Data directories configured at: /content/comma2k19_data


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


raw_data/Chunk_1.zip:   0%|          | 0.00/8.73G [00:00<?, ?B/s]

File verified! Size: 8.13 GB
Success! Data downloaded.
Found 7333 files in Chunk. Starting extraction...


Extracting Chunk: 100%|██████████| 7333/7333 [01:53<00:00, 64.69it/s]



Extraction complete! Files located at: /content/comma2k19_data/extracted
--- Processing Folder: b0c9d2329ad1606b_2018-07-29--11-17-20 ---
  Segment 3 : Steer [ -9.60,   9.90] | Speed [  2.47,  12.62] | Data points [Speed = 4975, Steer = 4974]
  Segment 4 : Steer [-72.90, 194.00] | Speed [  3.90,  22.29] | Data points [Speed = 4974, Steer = 4974]
  Segment 5 : Steer [ -7.50,   4.20] | Speed [ 22.26,  29.38] | Data points [Speed = 4973, Steer = 4973]
  Segment 6 : Steer [-11.10,   8.40] | Speed [ 26.70,  31.84] | Data points [Speed = 4973, Steer = 4973]
  Segment 7 : Steer [-10.20,   3.40] | Speed [ 24.95,  26.76] | Data points [Speed = 4974, Steer = 4974]


--- Processing Folder: b0c9d2329ad1606b_2018-08-02--16-41-38 ---
  Segment 5 : Steer [ -6.50,   6.30] | Speed [  0.00,  20.47] | Data points [Speed = 4974, Steer = 4975]
  Segment 6 : Steer [ -6.90,   9.10] | Speed [  0.00,  18.41] | Data points [Speed = 4975, Steer = 4974]
  Segment 7 : Steer [ -8.80,  18.70] | Speed [  0.00,  23.3

In [ ]:
!python comma2k19_FSD/convert_comma2k19_to_lerobot.py \
  --chunk-path "{CHUNK_PATH}" \
  --repo-id "local/comma2k19_act_smoke" \
  --output-root "{PROJECT_DIR / 'lerobot_datasets'}" \
  --max-episodes 2 \
  --max-frames-per-episode 64 \
  --overwrite

Map: 100% 64/64 [00:00<00:00, 1713.13 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[warn]: Preset M12 is mapped to M10.
Svt[info]: Level of Parallelism: 2
Svt[info]: Number of PPCS 42
Svt[info]: [asm level on system : up to avx512]
Svt[info]: [asm level selected : up to avx512]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 256 / 256 / 20 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 10 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-frame type 			: 2 / 16 / key frame
Svt[inf

## Convert Full Dataset

This writes the local LeRobotDataset used by ACT training. Use no history if you want state dimension 2, or use the default history offsets for richer state dimension 10.

In [ ]:
!python comma2k19_FSD/convert_comma2k19_to_lerobot.py \
  --chunk-path "{CHUNK_PATH}" \
  --repo-id "{DATASET_REPO_ID}" \
  --output-root "{PROJECT_DIR / 'lerobot_datasets'}" \
  --width 256 \
  --height 256 \
  --fps 20 \
  --future-time 1.0 \
  --max-episodes 20 \
  --overwrite

Map: 100% 1179/1179 [00:00<00:00, 1887.44 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[warn]: Preset M12 is mapped to M10.
Svt[info]: Level of Parallelism: 2
Svt[info]: Number of PPCS 42
Svt[info]: [asm level on system : up to avx512]
Svt[info]: [asm level selected : up to avx512]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 256 / 256 / 20 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 10 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-frame type 			: 2 / 16 / key frame
Svt

## Inspect Converted Dataset

In [ ]:
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
except ImportError:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset


dataset = LeRobotDataset(DATASET_REPO_ID, root=DATASET_ROOT)
print(dataset)
print(dataset.features)
sample = dataset[0]
print("state", sample["observation.state"].shape)
print("action", sample["action"].shape)
print("image", sample["observation.images.front"].shape)

LeRobotDataset({
    Repository ID: 'local/comma2k19_act',
    Number of selected episodes: '20',
    Number of selected samples: '23598',
    Features: '['observation.images.front', 'observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})
{'observation.images.front': {'dtype': 'video', 'shape': (256, 256, 3), 'names': ['height', 'width', 'channel'], 'info': {'video.height': 256, 'video.width': 256, 'video.codec': 'av1', 'video.pix_fmt': 'yuv420p', 'video.is_depth_map': False, 'video.fps': 20, 'video.channels': 3, 'has_audio': False}}, 'observation.state': {'dtype': 'float32', 'shape': (10,), 'names': ['speed', 'steer', 'speed_history_0', 'steer_history_0', 'speed_history_1', 'steer_history_1', 'speed_history_2', 'steer_history_2', 'speed_history_3', 'steer_history_3']}, 'action': {'dtype': 'float32', 'shape': (2,), 'names': ['future_speed', 'future_steer']}, 'timestamp': {'dtype': 'float32', 'shape': (1,), 'names': None}, 'frame_index': {

## Train ACT

ACT reads the observation/action dimensions from the saved LeRobotDataset metadata.

In [ ]:
!python comma2k19_FSD/train_act_policy.py \
  --dataset-repo-id "{DATASET_REPO_ID}" \
  --dataset-root "{DATASET_ROOT}" \
  --output-dir "{OUTPUT_DIR}" \
  --job-name act_comma2k19 \
  --device cuda \
  --batch-size 8 \
  --steps 10000 \
  --num-workers 4

Running ACT training command:
/usr/local/bin/lerobot-train --dataset.repo_id=local/comma2k19_act --dataset.root=/content/lerobot_datasets/local/comma2k19_act --policy.type=act --output_dir=/content/outputs/train/act_comma2k19 --job_name=act_comma2k19 --policy.device=cuda --batch_size=8 --steps=10000 --num_workers=4 --wandb.enable=false --policy.push_to_hub=false
INFO 2026-05-31 00:10:18 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                         

## Resume Training

After a run starts, LeRobot saves checkpoints under outputs/train/act_comma2k19/checkpoints.

In [ ]:
# Example resume command. Uncomment after a checkpoint exists.
# !lerobot-train \
#   --config_path "{OUTPUT_DIR / 'checkpoints' / 'last' / 'pretrained_model' / 'train_config.json'}" \
#   --resume=true

##Inference

In [ ]:
!python comma2k19_FSD/ACT_inference.py --chunk-path ./comma2k19_data/extracted/Chunk_1

Using local checkpoint: outputs/train/act_comma2k19/checkpoints/last/pretrained_model
Loading weights from local directory
Running inference with model: outputs/train/act_comma2k19/checkpoints/last/pretrained_model
Device: cuda
Using dataset: Comma_Instance
Expected visual keys: ['observation.images.front']
Expected state key: observation.state with dim=10
------------------------------
Sample 1:
  Input State (partial): tensor([32.9103, -0.7000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000])
  Predicted Action: [[0.8652409315109253, 0.00978856161236763]]
------------------------------
Sample 2:
  Input State (partial): tensor([33.2989, -4.9000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000])
  Predicted Action: [[0.8645351529121399, 0.011677823960781097]]
------------------------------
Sample 3:
  Input State (partial): tensor([32.8999, -1.6303,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.

In [ ]:
import lerobot
import lerobot.policies.act

print(dir(lerobot.policies.act))

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'configuration_act']


In [ ]:
import pkgutil
import lerobot.policies.act

for m in pkgutil.iter_modules(lerobot.policies.act.__path__):
    print(m.name)

configuration_act
modeling_act
processor_act


In [ ]:
from lerobot.policies.act import processor_act

print(dir(processor_act))

['ACTConfig', 'AddBatchDimensionProcessorStep', 'Any', 'DeviceProcessorStep', 'NormalizerProcessorStep', 'POLICY_POSTPROCESSOR_DEFAULT_NAME', 'POLICY_PREPROCESSOR_DEFAULT_NAME', 'PolicyAction', 'PolicyProcessorPipeline', 'RenameObservationsProcessorStep', 'UnnormalizerProcessorStep', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'make_act_pre_post_processors', 'policy_action_to_transition', 'torch', 'transition_to_policy_action']


In [ ]:
import inspect
from lerobot.policies.act.processor_act import make_act_pre_post_processors

print(inspect.signature(make_act_pre_post_processors))

(config: lerobot.policies.act.configuration_act.ACTConfig, dataset_stats: dict[str, dict[str, torch.Tensor]] | None = None) -> tuple[lerobot.processor.pipeline.DataProcessorPipeline[dict[str, typing.Any], dict[str, typing.Any]], lerobot.processor.pipeline.DataProcessorPipeline[torch.Tensor, torch.Tensor]]
